[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/salas/S2_quiz_ayer.ipynb)

# Quiz: lo de ayer, paso a paso

**Al terminar podrás** sacar a mano el MAE y el RMSE de cinco horas de MiBici,
y explicar por qué el RMSE sale más grande cuando hay un error grande.

De 15 a 20 minutos. No se entrega ni se califica. Son tres partes:

| Parte | Qué hacen | Minutos |
|---|---|---|
| 1. Conceptos de ayer | 10 preguntas cortas | 5 |
| 2. El error en Python | cinco horas de la estación 51, con pandas | 8 |
| 3. Qué significa cada número | 5 preguntas de operaciones | 4 |

**Cómo trabajar.** Corran las celdas en orden. Cada pregunta tiene opciones a), b), c) o d):
decidan su respuesta y escriban la letra debajo de la pregunta. En la parte 2, hagan la cuenta a mano antes
de correr la celda de Python que la comprueba.

Corran las celdas de abajo hasta que aparezca `✅ entorno listo`.

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, mean_squared_error

## Parte 1. Conceptos de ayer (5 min)

Diez preguntas cortas. Todas salen de lo que vimos ayer y del repaso de hace un momento.

**1.1 ¿Qué es el costo J?**

- a) El número de filas de train
- b) Qué tan mal ajusta una recta: la mitad del promedio de los errores al cuadrado
- c) El porcentaje de horas en que la recta acierta

**Su respuesta (doble clic para escribir):** ___

**1.2 Al ajustar (entrenar) una recta, ¿qué buscamos?**

- a) Los w y b con el costo J más grande
- b) Una recta que pase exacto por todos los puntos
- c) Los w y b con el costo J más chico

**Su respuesta (doble clic para escribir):** ___

**1.3 ¿Cómo se llama el algoritmo que baja por el costo paso a paso?**

- a) Descenso de gradiente
- b) Validación
- c) Valor absoluto

**Su respuesta (doble clic para escribir):** ___

**1.4 LinearRegression de scikit-learn, ¿cómo encuentra w y b?**

- a) Prueba valores de w al azar hasta que le gusta uno
- b) Resuelve los mínimos cuadrados de un salto, con una cuenta directa (el atajo de la ecuación normal)
- c) Con descenso de gradiente, mil pasos chiquitos

**Su respuesta (doble clic para escribir):** ___

**1.5 En la recta ŷ = w x + b, ¿qué es w?**

- a) La pendiente: cuánto sube la predicción por cada unidad de x
- b) La predicción
- c) El error de la fila

**Su respuesta (doble clic para escribir):** ___

**1.6 La recta de ayer es ŷ = 0.70 x + 1.18 (b = 1.18 es la base, lo que dice cuando x vale 0). Si hace una semana salieron 10 bicis, ¿cuánto dice la recta?**

- a) Unas 8 bicis
- b) 10 bicis
- c) Unas 12 bicis

**Su respuesta (doble clic para escribir):** ___

**1.7 La regresión polinomial dibuja una curva. ¿Por qué se sigue llamando regresión lineal?**

- a) Porque la gráfica en realidad es una recta
- b) Porque es lineal en los pesos: inventamos columnas x², x³ y seguimos sumando columnas por pesos
- c) Porque solo se puede usar con grado 1

**Su respuesta (doble clic para escribir):** ___

**1.8 ¿Qué es train y qué es validación (julio)?**

- a) Son lo mismo con otro nombre
- b) Train es con lo que se ajusta el modelo; validación es con lo que se mide, y el modelo no la vio
- c) Train es julio y validación es octubre a junio

**Su respuesta (doble clic para escribir):** ___

**1.9 ¿Qué es un residuo?**

- a) El promedio de todos los errores
- b) Las filas que sobran después de partir los datos
- c) El error de una fila: lo real menos lo predicho

**Su respuesta (doble clic para escribir):** ___

**1.10 ¿Por qué no resumimos los errores sumándolos tal cual?**

- a) Porque los de arriba se cancelan con los de abajo
- b) Porque la suma tarda mucho en calcularse
- c) Porque la suma siempre da negativa

**Su respuesta (doble clic para escribir):** ___

## Parte 2. El error, programado en Python con pandas (8 min)

Las mismas cinco horas del repaso: estación 51, lunes 27 de julio, de las 8:00 a las 12:00.
La predicción es la regla "las mismas bicis que hace una semana".

| Hora | Hace una semana (predicción, ŷ) | Salieron de verdad (y) |
|---|---|---|
| 8:00 | 32 | 34 |
| 9:00 | 20 | 22 |
| 10:00 | 8 | 10 |
| 11:00 | 11 | 12 |
| 12:00 | 11 | 4 |

Ahora lo programan ustedes en Python, con pandas, columna por columna. Son 5 filas: pueden revisar cada cuenta con la vista.
En cada celda falta una sola palabra o número en cada `___`. Córranla y escriban la letra de la pregunta en la hoja.

**Lo único de pandas que necesitan** (ejemplo con otra tabla, la de un puesto de tacos):

| Quiero... | Se escribe | Con `t.ventas` = 3, −1, 2 da |
|---|---|---|
| restar dos columnas | `t.ventas - t.meta` | una columna nueva, fila por fila |
| quitar el signo | `t.ventas.abs()` | 3, 1, 2 |
| elevar al cuadrado | `t.ventas ** 2` | 9, 1, 4 |
| sumar la columna | `t.ventas.sum()` | 4 |
| sacar el promedio | `t.ventas.mean()` | 1.33 |
| sacar raíz | `numero ** 0.5` | la raíz de 9 es 3 |

In [ ]:
tabla = pd.DataFrame({"hora": [8, 9, 10, 11, 12],
                      "predicho": [32, 20, 8, 11, 11],   # la regla: lo de hace una semana
                      "real": [34, 22, 10, 12, 4]})      # lo que salió de verdad
tabla

### Paso 1. El residuo de cada fila: real menos predicho

**2.1 A las 8:00 salieron 34 y la regla dijo 32. ¿Cuál es el residuo (real menos predicho)?**

- a) 2
- b) -2
- c) 66

**Su respuesta (doble clic para escribir):** ___

**2.2 A las 12:00 salieron 4 y la regla dijo 11. ¿Cuál es el residuo?**

- a) 7
- b) -7
- c) 15

**Su respuesta (doble clic para escribir):** ___

In [ ]:
tabla["residuo"] = tabla.___ - tabla.___   # completen: real menos predicho
tabla

### Paso 2. La suma con signo

**2.3 Los residuos son 2, 2, 2, 1 y -7. ¿Cuánto suman?**

- a) 14
- b) 0
- c) 7

**Su respuesta (doble clic para escribir):** ___

In [ ]:
suma = tabla.residuo.___()   # completen: la función que suma
print("suma con signo:", suma)

### Paso 3. Sin signo: valores absolutos y MAE

**2.4 Quitando el signo, los errores son 2, 2, 2, 1 y 7. ¿Cuánto suman?**

- a) 0
- b) 14
- c) 62

**Su respuesta (doble clic para escribir):** ___

**2.5 El MAE es el promedio de esos valores absolutos: su suma entre las 5 horas. ¿Cuánto da?**

- a) 2.8
- b) 14
- c) 3.5

**Su respuesta (doble clic para escribir):** ___

In [ ]:
tabla["absoluto"] = tabla.residuo.___()   # completen: la función que quita el signo
mae = tabla.absoluto.___()                 # completen: la función del promedio
print("suma sin signo:", tabla.absoluto.sum(), "  MAE:", mae)

### Paso 4. Al cuadrado: MSE

**2.6 ¿Cuánto es el residuo -7 al cuadrado?**

- a) -49
- b) 49
- c) 14

**Su respuesta (doble clic para escribir):** ___

**2.7 Eleven al cuadrado los cinco residuos (2, 2, 2, 1 y -7). ¿Cuánto suman los cuadrados?**

- a) 14
- b) 49
- c) 62

**Su respuesta (doble clic para escribir):** ___

**2.8 El MSE es el promedio de los cuadrados: su suma entre las 5 horas. ¿Cuánto da?**

- a) 2.8
- b) 12.4
- c) 62

**Su respuesta (doble clic para escribir):** ___

In [ ]:
tabla["cuadrado"] = tabla.residuo ** ___   # completen: el exponente del cuadrado
sse = tabla.cuadrado.___()                 # completen: SSE es la suma
mse = tabla.cuadrado.___()                 # completen: MSE es el promedio
print("SSE:", sse, "  MSE:", mse)

### Paso 5. La raíz: RMSE

**2.9 El RMSE es la raíz cuadrada del MSE: raíz de 12.4. ¿Cuál se acerca más? (Pista: 3.5 por 3.5 es 12.25.)**

- a) Unas 2.8 bicis
- b) Unas 3.5 bicis
- c) Unas 6.2 bicis
- d) 12.4 bicis

**Su respuesta (doble clic para escribir):** ___

In [ ]:
rmse = mse ** ___   # completen: la raíz es elevar a un medio
print(f"RMSE: {rmse:.2f}")
tabla

Para comprobar con scikit-learn (opcional): `mean_absolute_error(tabla.real, tabla.predicho)` y `root_mean_squared_error(tabla.real, tabla.predicho)` deben dar lo mismo que sus columnas.

### Paso 6. MAE contra RMSE

**2.10 Con las mismas cinco horas salió MAE 2.8 y RMSE 3.5. ¿Por qué el RMSE es mayor?**

- a) Porque el RMSE cuenta más horas que el MAE
- b) Porque al elevar al cuadrado, el error grande pesa más: el 7 vale 49 y los 2 valen 4
- c) Porque hay un error en la cuenta

**Su respuesta (doble clic para escribir):** ___

**2.11 Si quitamos la hora de las 12:00, quedan errores 2, 2, 2 y 1: MAE 1.75 y RMSE 1.80. ¿Qué cambió?**

- a) Sin el error grande, MAE y RMSE casi coinciden
- b) El RMSE ahora es más chico que el MAE
- c) Nada: la diferencia sigue igual de grande

**Su respuesta (doble clic para escribir):** ___

In [ ]:
sin12 = tabla[tabla.hora < 12]   # las 4 horas sin el error grande
mae_sin = sin12.absoluto.___()               # completen: igual que el MAE de arriba
rmse_sin = sin12.cuadrado.___() ** ___       # completen: igual que el RMSE de arriba
print(f"Sin las 12:00: MAE {mae_sin:.2f}   RMSE {rmse_sin:.2f}")

## Parte 3. Qué significa cada número (4 min)

Ahora con los números de todo julio, los del repaso:

| | MAE | RMSE | R² |
|---|---|---|---|
| La regla (hace una semana) | 2.30 | 4.16 | 0.41 |
| La recta | 2.35 | 3.84 | 0.50 |

**3.1 La regla tiene MAE 2.30 en julio. Para operaciones quiere decir...**

- a) Que acierta el 2.3 % de las horas
- b) Que en una hora típica de julio falla por unas 2.3 bicis, para arriba o para abajo
- c) Que su peor error fue de 2.3 bicis

**Su respuesta (doble clic para escribir):** ___

**3.2 La regla tiene RMSE 4.16 y MAE 2.30. Que el RMSE sea bastante mayor quiere decir...**

- a) Que hay algunas horas con errores grandes, que el cuadrado infla
- b) Que la regla está mal programada
- c) Que hay que reportar siempre el número más chico

**Su respuesta (doble clic para escribir):** ___

**3.3 La recta tiene R² 0.50 en julio. Quiere decir...**

- a) Que acierta la mitad de las horas
- b) Que falla por media bici
- c) Que su error al cuadrado es la mitad del que tendría si dijera siempre el promedio de julio

**Su respuesta (doble clic para escribir):** ___

**3.4 A operaciones le duelen mucho los errores grandes (una estación vacía en hora pico). ¿Cuál reporto?**

- a) El MAE
- b) El RMSE
- c) El MAPE

**Su respuesta (doble clic para escribir):** ___

**3.5 En julio, 29 de cada 100 horas tienen 0 salidas. ¿Qué le pasa al MAPE (error en porcentaje)?**

- a) Funciona igual que el MAE
- b) Sale 0 en esas horas
- c) Se dispara, porque divide entre lo real y lo real es 0

**Su respuesta (doble clic para escribir):** ___